In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import sklearn
import lightgbm as lgb
from sklearn.model_selection import KFold
import random as rn
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
rn.seed(RANDOM_SEED)
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [ ]:
import pathlib

data_path = pathlib.Path().cwd().parent/'data'
data_path

In [3]:
train_whole_data = pd.read_parquet(data_path/'whole_data.parquet')
train_whole_data.head()



,building_id,building_type,datetime_,temperature_celsius,rainfall_mm,windspeed_mps,humidity_percent,sunshine_hour,solar_radiation_wpm2,power_consumption_kwh,total_area_m2,cooling_area_m2,solar_panel_capacity_kw,ess_storage_capacity_kwh,pcs_capacity_kw
0,62,연구소,2024-07-03 03:00:00,21.5,0.0,1.2,95.0,0.0,0.000000,1445.40,66240.0,31867.0,NaN,NaN,NaN
1,62,연구소,2024-07-03 04:00:00,22.5,0.0,0.3,94.0,0.0,0.000000,1481.40,66240.0,31867.0,NaN,NaN,NaN
2,62,연구소,2024-07-03 05:00:00,23.0,0.0,1.2,94.0,0.0,0.000000,1450.08,66240.0,31867.0,NaN,NaN,NaN
3,62,연구소,2024-07-03 06:00:00,24.3,0.0,1.5,89.0,0.0,16.666667,1521.18,66240.0,31867.0,NaN,NaN,NaN
4,62,연구소,2024-07-03 07:00:00,26.0,0.0,3.4,82.0,0.0,55.555556,1804.14,66240.0,31867.0,NaN,NaN,NaN


In [80]:
# 건물 유형별 평균 전력 사용량 조회
import duckdb
power_by_type = duckdb.query("""
    SELECT
        building_type,
        AVG(power_consumption_kwh) AS avg_power_consumption
    FROM train_whole_data
    GROUP BY building_type
    ORDER BY avg_power_consumption DESC;
""").to_df()

print("\n📊 건물 유형별 평균 전력 사용량:")
print(power_by_type)


📊 건물 유형별 평균 전력 사용량:
  building_type  avg_power_consumption
0      IDC(전화국)           10316.944618
1            병원            4454.059004
2            학교            3462.680492
3            호텔            3175.016948
4           백화점            2729.738252
5            상용            2513.698044
6          건물기타            2285.963948
7           연구소            2111.671770
8            공공            1625.914862
9           아파트            1106.309105


In [81]:
# 시간대별 평균 전력 사용량 조회
power_by_hour = duckdb.query("""
    SELECT
        EXTRACT(hour FROM datetime_) AS hour_of_day,
        AVG(power_consumption_kwh) AS avg_power_consumption
    FROM train_whole_data
    GROUP BY hour_of_day
    ORDER BY hour_of_day;
""").to_df()

print("\n📊 시간대별 평균 전력 사용량:")
print(power_by_hour)


📊 시간대별 평균 전력 사용량:
    hour_of_day  avg_power_consumption
0             0            2849.762576
1             1            2774.322140
2             2            2719.511346
3             3            2680.842655
4             4            2654.102702
5             5            2662.018008
6             6            2764.066234
7             7            2950.034984
8             8            3164.467151
9             9            3491.010189
10           10            3818.715674
11           11            3899.723016
12           12            3912.607069
13           13            3946.149716
14           14            3963.308251
15           15            3974.968867
16           16            3956.181640
17           17            3901.863020
18           18            3741.598178
19           19            3599.263378
20           20            3406.713298
21           21            3138.923612
22           22            3001.850433
23           23            2937.816431


In [82]:
train_whole_data.head()

,building_id,building_type,datetime_,temperature_celsius,rainfall_mm,windspeed_mps,humidity_percent,sunshine_hour,solar_radiation_wpm2,power_consumption_kwh,total_area_m2,cooling_area_m2,solar_panel_capacity_kw,ess_storage_capacity_kwh,pcs_capacity_kw
0,62,연구소,2024-07-03 03:00:00,21.5,0.0,1.2,95.0,0.0,0.000000,1445.40,66240.0,31867.0,NaN,NaN,NaN
1,62,연구소,2024-07-03 04:00:00,22.5,0.0,0.3,94.0,0.0,0.000000,1481.40,66240.0,31867.0,NaN,NaN,NaN
2,62,연구소,2024-07-03 05:00:00,23.0,0.0,1.2,94.0,0.0,0.000000,1450.08,66240.0,31867.0,NaN,NaN,NaN
3,62,연구소,2024-07-03 06:00:00,24.3,0.0,1.5,89.0,0.0,16.666667,1521.18,66240.0,31867.0,NaN,NaN,NaN
4,62,연구소,2024-07-03 07:00:00,26.0,0.0,3.4,82.0,0.0,55.555556,1804.14,66240.0,31867.0,NaN,NaN,NaN


In [83]:
# 전처리 쿼리 실행하여 새로운 데이터프레임 생성
train_preprocessed_df = duckdb.query("""
    SELECT
    *,
        
        -- 1. 시간 관련 피처 생성
        EXTRACT(month FROM datetime_) AS month,
        EXTRACT(day FROM datetime_) AS day,
        EXTRACT(hour FROM datetime_) AS hour,
        EXTRACT(dayofweek FROM datetime_) AS dayofweek,
        
        -- 2. 주말 피처 생성 (토요일=6, 일요일=0 in DuckDB's convention)
        CASE 
            WHEN EXTRACT(dayofweek FROM datetime_) IN (0, 6) THEN 1 
            ELSE 0 
        END AS is_weekend,
        
        -- 타겟 변수
        power_consumption_kwh
        
    FROM train_whole_data;
""").to_df()

print("\n⚙️ DuckDB로 전처리 완료!")
print(train_preprocessed_df.head())


⚙️ DuckDB로 전처리 완료!
   building_id building_type           datetime_  temperature_celsius  \
0           62           연구소 2024-07-03 03:00:00                 21.5   
1           62           연구소 2024-07-03 04:00:00                 22.5   
2           62           연구소 2024-07-03 05:00:00                 23.0   
3           62           연구소 2024-07-03 06:00:00                 24.3   
4           62           연구소 2024-07-03 07:00:00                 26.0   

   rainfall_mm  windspeed_mps  humidity_percent  sunshine_hour  \
0          0.0            1.2              95.0            0.0   
1          0.0            0.3              94.0            0.0   
2          0.0            1.2              94.0            0.0   
3          0.0            1.5              89.0            0.0   
4          0.0            3.4              82.0            0.0   

   solar_radiation_wpm2  power_consumption_kwh  total_area_m2  \
0              0.000000                1445.40        66240.0   
1             

In [84]:
train_preprocessed_df = train_preprocessed_df.drop('power_consumption_kwh_1', axis=1)
train_preprocessed_df.head()


,building_id,building_type,datetime_,temperature_celsius,rainfall_mm,windspeed_mps,humidity_percent,sunshine_hour,solar_radiation_wpm2,power_consumption_kwh,total_area_m2,cooling_area_m2,solar_panel_capacity_kw,ess_storage_capacity_kwh,pcs_capacity_kw,month,day,hour,dayofweek,is_weekend
0,62,연구소,2024-07-03 03:00:00,21.5,0.0,1.2,95.0,0.0,0.000000,1445.40,66240.0,31867.0,NaN,NaN,NaN,7,3,3,3,0
1,62,연구소,2024-07-03 04:00:00,22.5,0.0,0.3,94.0,0.0,0.000000,1481.40,66240.0,31867.0,NaN,NaN,NaN,7,3,4,3,0
2,62,연구소,2024-07-03 05:00:00,23.0,0.0,1.2,94.0,0.0,0.000000,1450.08,66240.0,31867.0,NaN,NaN,NaN,7,3,5,3,0
3,62,연구소,2024-07-03 06:00:00,24.3,0.0,1.5,89.0,0.0,16.666667,1521.18,66240.0,31867.0,NaN,NaN,NaN,7,3,6,3,0
4,62,연구소,2024-07-03 07:00:00,26.0,0.0,3.4,82.0,0.0,55.555556,1804.14,66240.0,31867.0,NaN,NaN,NaN,7,3,7,3,0


In [85]:
train_preprocessed_df.isnull().sum()

building_id                      0
building_type                    0
datetime_                        0
temperature_celsius              0
rainfall_mm                      0
windspeed_mps                    0
humidity_percent                 0
sunshine_hour                    0
solar_radiation_wpm2             0
power_consumption_kwh            0
total_area_m2                    0
cooling_area_m2                  0
solar_panel_capacity_kw      95880
ess_storage_capacity_kwh    179520
pcs_capacity_kw             179520
month                            0
day                              0
hour                             0
dayofweek                        0
is_weekend                       0
dtype: int64

In [86]:
# 결측치를 0으로 채울 컬럼 목록
cols_to_fill_zero = ['solar_panel_capacity_kw', 'ess_storage_capacity_kwh', 'pcs_capacity_kw']

# 지정된 컬럼의 NaN 값을 0으로 바꿉니다.
train_preprocessed_df[cols_to_fill_zero] = train_preprocessed_df[cols_to_fill_zero].fillna(0)

print("✅ 결측치 처리가 완료되었습니다.")

✅ 결측치 처리가 완료되었습니다.


In [87]:
# --- Lag 피처 ---
# 24시간 전의 전력 사용량을 새로운 피처로 추가합니다.
train_preprocessed_df['power_lag_24h'] = train_preprocessed_df.groupby('building_id')['power_consumption_kwh'].shift(24)

# --- Rolling 피처 ---
# 건물별로 최근 7일간의 기온 이동 평균을 계산합니다.
train_preprocessed_df['temp_rolling_mean_7d'] = train_preprocessed_df.groupby('building_id')['temperature_celsius'].rolling(window=24*7, min_periods=1).mean().reset_index(drop=True)

# 이 과정에서 생긴 결측치(NaN)는 0으로 채웁니다.
train_preprocessed_df = train_preprocessed_df.fillna(0)

print("✅ Lag 및 Rolling 피처 생성이 완료되었습니다.")

train_preprocessed_df.head()

✅ Lag 및 Rolling 피처 생성이 완료되었습니다.


,building_id,building_type,datetime_,temperature_celsius,rainfall_mm,windspeed_mps,humidity_percent,sunshine_hour,solar_radiation_wpm2,power_consumption_kwh,total_area_m2,cooling_area_m2,solar_panel_capacity_kw,ess_storage_capacity_kwh,pcs_capacity_kw,month,day,hour,dayofweek,is_weekend,power_lag_24h,temp_rolling_mean_7d
0,62,연구소,2024-07-03 03:00:00,21.5,0.0,1.2,95.0,0.0,0.000000,1445.40,66240.0,31867.0,0.0,0.0,0.0,7,3,3,3,0,0.0,18.300000
1,62,연구소,2024-07-03 04:00:00,22.5,0.0,0.3,94.0,0.0,0.000000,1481.40,66240.0,31867.0,0.0,0.0,0.0,7,3,4,3,0,0.0,18.300000
2,62,연구소,2024-07-03 05:00:00,23.0,0.0,1.2,94.0,0.0,0.000000,1450.08,66240.0,31867.0,0.0,0.0,0.0,7,3,5,3,0,0.0,18.233333
3,62,연구소,2024-07-03 06:00:00,24.3,0.0,1.5,89.0,0.0,16.666667,1521.18,66240.0,31867.0,0.0,0.0,0.0,7,3,6,3,0,0.0,18.175000
4,62,연구소,2024-07-03 07:00:00,26.0,0.0,3.4,82.0,0.0,55.555556,1804.14,66240.0,31867.0,0.0,0.0,0.0,7,3,7,3,0,0.0,18.100000


In [88]:
# --- 불쾌지수 ---
# 습도(%)를 비율(0~1)로 변환합니다.
train_preprocessed_df['humidity_ratio'] = train_preprocessed_df['humidity_percent'] / 100.0
# 간단한 형태의 불쾌지수를 생성합니다.
train_preprocessed_df['discomfort_index'] = train_preprocessed_df['temperature_celsius'] - 0.55 * (1 - train_preprocessed_df['humidity_ratio']) * (train_preprocessed_df['temperature_celsius'] - 14.5)

# --- 시간과 기온의 상호작용 피처 ---
# 주기성 피처 생성 후, 아래 코드의 주석을 해제하여 사용할 수 있습니다.
# train_preprocessed_df['temp_x_hour_cos'] = train_preprocessed_df['temperature_celsius'] * train_preprocessed_df['hour_cos']


print("✅ 상호작용 및 파생 피처 생성이 완료되었습니다.")
train_preprocessed_df.head()

✅ 상호작용 및 파생 피처 생성이 완료되었습니다.


,building_id,building_type,datetime_,temperature_celsius,rainfall_mm,windspeed_mps,humidity_percent,sunshine_hour,solar_radiation_wpm2,power_consumption_kwh,total_area_m2,cooling_area_m2,solar_panel_capacity_kw,ess_storage_capacity_kwh,pcs_capacity_kw,month,day,hour,dayofweek,is_weekend,power_lag_24h,temp_rolling_mean_7d,humidity_ratio,discomfort_index
0,62,연구소,2024-07-03 03:00:00,21.5,0.0,1.2,95.0,0.0,0.000000,1445.40,66240.0,31867.0,0.0,0.0,0.0,7,3,3,3,0,0.0,18.300000,0.95,21.3075
1,62,연구소,2024-07-03 04:00:00,22.5,0.0,0.3,94.0,0.0,0.000000,1481.40,66240.0,31867.0,0.0,0.0,0.0,7,3,4,3,0,0.0,18.300000,0.94,22.2360
2,62,연구소,2024-07-03 05:00:00,23.0,0.0,1.2,94.0,0.0,0.000000,1450.08,66240.0,31867.0,0.0,0.0,0.0,7,3,5,3,0,0.0,18.233333,0.94,22.7195
3,62,연구소,2024-07-03 06:00:00,24.3,0.0,1.5,89.0,0.0,16.666667,1521.18,66240.0,31867.0,0.0,0.0,0.0,7,3,6,3,0,0.0,18.175000,0.89,23.7071
4,62,연구소,2024-07-03 07:00:00,26.0,0.0,3.4,82.0,0.0,55.555556,1804.14,66240.0,31867.0,0.0,0.0,0.0,7,3,7,3,0,0.0,18.100000,0.82,24.8615


In [89]:
# 1. 2024년 대한민국 주요 공휴일 및 대체 공휴일 목록을 정의합니다.
holidays_2024 = [
    '2024-01-01', # 신정
    '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12', # 설날 연휴
    '2024-03-01', # 삼일절
    '2024-05-06', # 어린이날 대체공휴일
    '2024-05-15', # 부처님 오신 날
    '2024-06-06', # 현충일
    '2024-08-15', # 광복절
    '2024-09-16', '2024-09-17', '2024-09-18', # 추석 연휴
    '2024-10-03', # 개천절
    '2024-10-09', # 한글날
    '2024-12-25', # 크리스마스
]

# 2. 문자열 리스트를 Pandas가 인식할 수 있는 Datetime 형식으로 변환합니다.
holidays_dt = pd.to_datetime(holidays_2024)

# 3. 'train_preprocessed_df'의 'datetime_' 컬럼을 사용하여 'is_holiday' 컬럼을 생성합니다.
train_preprocessed_df['is_holiday'] = train_preprocessed_df['datetime_'].dt.normalize().isin(holidays_dt).astype(int)

print("✅ 공휴일 피처('is_holiday') 생성이 완료되었습니다.")

# 4. 피처가 잘 생성되었는지 확인합니다.
print(f"\n생성된 공휴일 피처의 총 개수: {train_preprocessed_df['is_holiday'].sum()}개")
print("\n'is_holiday' 컬럼이 추가된 데이터 샘플:")
print(train_preprocessed_df[['datetime_', 'is_holiday']].head())

train_preprocessed_df.head()

✅ 공휴일 피처('is_holiday') 생성이 완료되었습니다.

생성된 공휴일 피처의 총 개수: 4800개

'is_holiday' 컬럼이 추가된 데이터 샘플:
            datetime_  is_holiday
0 2024-07-03 03:00:00           0
1 2024-07-03 04:00:00           0
2 2024-07-03 05:00:00           0
3 2024-07-03 06:00:00           0
4 2024-07-03 07:00:00           0


,building_id,building_type,datetime_,temperature_celsius,rainfall_mm,windspeed_mps,humidity_percent,sunshine_hour,solar_radiation_wpm2,power_consumption_kwh,total_area_m2,cooling_area_m2,solar_panel_capacity_kw,ess_storage_capacity_kwh,pcs_capacity_kw,month,day,hour,dayofweek,is_weekend,power_lag_24h,temp_rolling_mean_7d,humidity_ratio,discomfort_index,is_holiday
0,62,연구소,2024-07-03 03:00:00,21.5,0.0,1.2,95.0,0.0,0.000000,1445.40,66240.0,31867.0,0.0,0.0,0.0,7,3,3,3,0,0.0,18.300000,0.95,21.3075,0
1,62,연구소,2024-07-03 04:00:00,22.5,0.0,0.3,94.0,0.0,0.000000,1481.40,66240.0,31867.0,0.0,0.0,0.0,7,3,4,3,0,0.0,18.300000,0.94,22.2360,0
2,62,연구소,2024-07-03 05:00:00,23.0,0.0,1.2,94.0,0.0,0.000000,1450.08,66240.0,31867.0,0.0,0.0,0.0,7,3,5,3,0,0.0,18.233333,0.94,22.7195,0
3,62,연구소,2024-07-03 06:00:00,24.3,0.0,1.5,89.0,0.0,16.666667,1521.18,66240.0,31867.0,0.0,0.0,0.0,7,3,6,3,0,0.0,18.175000,0.89,23.7071,0
4,62,연구소,2024-07-03 07:00:00,26.0,0.0,3.4,82.0,0.0,55.555556,1804.14,66240.0,31867.0,0.0,0.0,0.0,7,3,7,3,0,0.0,18.100000,0.82,24.8615,0


In [90]:
import pandas as pd

# 1. 원본 'building_type'을 새로운 컬럼('building_type_original')에 복사합니다.
train_preprocessed_df['building_type_original'] = train_preprocessed_df['building_type']

# 2. 이제 get_dummies를 실행합니다.
# 'building_type' 컬럼은 새로운 인코딩된 컬럼들로 대체되지만,
# 복사해 둔 'building_type_original'은 그대로 남아있게 됩니다.
train_preprocessed_df = pd.get_dummies(train_preprocessed_df, columns=['building_type'])

print("✅ 원본 컬럼을 유지한 채 원-핫 인코딩이 완료되었습니다.")

# 'building_type_original' 컬럼과 새롭게 생성된 인코딩 컬럼들을 확인합니다.
print(train_preprocessed_df.filter(regex='building_type').head())

✅ 원본 컬럼을 유지한 채 원-핫 인코딩이 완료되었습니다.
  building_type_original  building_type_IDC(전화국)  building_type_건물기타  \
0                    연구소                   False               False   
1                    연구소                   False               False   
2                    연구소                   False               False   
3                    연구소                   False               False   
4                    연구소                   False               False   

   building_type_공공  building_type_백화점  building_type_병원  building_type_상용  \
0             False              False             False             False   
1             False              False             False             False   
2             False              False             False             False   
3             False              False             False             False   
4             False              False             False             False   

   building_type_아파트  building_type_연구소  building_type_학교  buildi

In [91]:
train_preprocessed_df.head()

,building_id,datetime_,temperature_celsius,rainfall_mm,windspeed_mps,humidity_percent,sunshine_hour,solar_radiation_wpm2,power_consumption_kwh,total_area_m2,cooling_area_m2,solar_panel_capacity_kw,ess_storage_capacity_kwh,pcs_capacity_kw,month,day,hour,dayofweek,is_weekend,power_lag_24h,temp_rolling_mean_7d,humidity_ratio,discomfort_index,is_holiday,building_type_original,building_type_IDC(전화국),building_type_건물기타,building_type_공공,building_type_백화점,building_type_병원,building_type_상용,building_type_아파트,building_type_연구소,building_type_학교,building_type_호텔
0,62,2024-07-03 03:00:00,21.5,0.0,1.2,95.0,0.0,0.000000,1445.40,66240.0,31867.0,0.0,0.0,0.0,7,3,3,3,0,0.0,18.300000,0.95,21.3075,0,연구소,False,False,False,False,False,False,False,True,False,False
1,62,2024-07-03 04:00:00,22.5,0.0,0.3,94.0,0.0,0.000000,1481.40,66240.0,31867.0,0.0,0.0,0.0,7,3,4,3,0,0.0,18.300000,0.94,22.2360,0,연구소,False,False,False,False,False,False,False,True,False,False
2,62,2024-07-03 05:00:00,23.0,0.0,1.2,94.0,0.0,0.000000,1450.08,66240.0,31867.0,0.0,0.0,0.0,7,3,5,3,0,0.0,18.233333,0.94,22.7195,0,연구소,False,False,False,False,False,False,False,True,False,False
3,62,2024-07-03 06:00:00,24.3,0.0,1.5,89.0,0.0,16.666667,1521.18,66240.0,31867.0,0.0,0.0,0.0,7,3,6,3,0,0.0,18.175000,0.89,23.7071,0,연구소,False,False,False,False,False,False,False,True,False,False
4,62,2024-07-03 07:00:00,26.0,0.0,3.4,82.0,0.0,55.555556,1804.14,66240.0,31867.0,0.0,0.0,0.0,7,3,7,3,0,0.0,18.100000,0.82,24.8615,0,연구소,False,False,False,False,False,False,False,True,False,False


In [92]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from tqdm import tqdm # 모델 학습 진행 상황을 시각적으로 보여주는 라이브러리

# --- 1. SMAPE 평가 함수 정의 ---
def smape(y_true, y_pred):
    """SMAPE(Symmetric Mean Absolute Percentage Error)를 계산하는 함수"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    numerator = np.abs(y_pred - y_true)
    denominator = np.abs(y_true) + np.abs(y_pred)
    fraction = np.divide(numerator, denominator, out=np.zeros_like(numerator, dtype=float), where=denominator!=0)
    return np.mean(fraction) * 100

# --- 2. 건물별 모델 학습 및 평가 ---

# 전체 예측값과 실제값을 저장할 리스트를 초기화합니다.
all_predictions = []
all_true_values = []
building_scores = []

# 'train_preprocessed_df'에 있는 모든 고유한 building_id에 대해 반복합니다.
unique_building_ids = train_preprocessed_df['building_id'].unique()

for building_id in tqdm(unique_building_ids, desc="건물별 모델 학습 진행률"):
    
    # 해당 건물의 데이터만 필터링합니다.
    building_data = train_preprocessed_df[train_preprocessed_df['building_id'] == building_id].copy()

    # 데이터가 너무 적으면(예: 50개 미만) 학습이 어려우므로 건너뜁니다.
    if len(building_data) < 50:
        continue

    # 1. 피처(X)와 타겟(y)을 정의합니다.
    target = 'power_consumption_kwh'
    # 모델 학습에 불필요한 컬럼들을 제외합니다.
    features_to_exclude = [target, 'datetime_', 'building_type_original']
    features = [col for col in building_data.columns if col not in features_to_exclude]
    
    X = building_data[features]
    y = building_data[target]

    # 2. 각 건물의 데이터를 8:2로 분할합니다 (시간 순서 유지).
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False, random_state=42
    )
    
    # 3. 새로운 LightGBM 모델을 생성하고 학습합니다.
    model = lgb.LGBMRegressor(random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # 4. 검증 데이터로 예측을 수행합니다.
    preds = model.predict(X_test)

    # 현재 건물의 SMAPE 점수를 계산하고 저장합니다.
    score = smape(y_test, preds)
    building_scores.append({'building_id': building_id, 'smape': score})

    # 전체 점수 계산을 위해 예측값과 실제값을 저장합니다.
    all_predictions.extend(preds)
    all_true_values.extend(y_test)

# --- 3. 최종 결과 확인 ---
print("\n" + "="*50)

# 전체 데이터에 대한 최종 SMAPE 점수를 계산합니다.
overall_smape = smape(all_true_values, all_predictions)
print(f"✅ 전체 건물의 통합 SMAPE 점수: {overall_smape:.4f}")
print("="*50)

# 개별 건물 점수를 데이터프레임으로 변환하여 확인합니다.
building_scores_df = pd.DataFrame(building_scores).sort_values('smape', ascending=False)

print("\n--- 🎯 예측 오차가 가장 큰 건물 Top 5 ---")
print(building_scores_df.head(5))

print("\n--- ✅ 예측 오차가 가장 작은 건물 Top 5 ---")
print(building_scores_df.tail(5))

건물별 모델 학습 진행률:   2%|▏         | 2/100 [00:00<00:05, 17.59it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000132 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1485
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start training from score 1821.917647
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000106 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1500
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start training from score 1200.183311
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000182 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1500
[LightGBM] [Info] Number of data points in the train

건물별 모델 학습 진행률:   6%|▌         | 6/100 [00:00<00:05, 17.29it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000147 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1493
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start training from score 2077.279603
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000123 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1490
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start training from score 10634.974077
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000214 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1506
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start

건물별 모델 학습 진행률:  10%|█         | 10/100 [00:00<00:04, 18.10it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000128 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1462
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start training from score 1751.650576
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000120 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1500
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start training from score 3276.962351
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000122 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1500
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start 

건물별 모델 학습 진행률:  97%|█████████▋| 97/100 [00:04<00:00, 23.82it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000147 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1474
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start training from score 5543.974117
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000129 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1492
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start training from score 17522.446948
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000163 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1475
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start

건물별 모델 학습 진행률: 100%|██████████| 100/100 [00:04<00:00, 21.72it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000190 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1462
[LightGBM] [Info] Number of data points in the train set: 1632, number of used features: 16
[LightGBM] [Info] Start training from score 1741.848234

✅ 전체 건물의 통합 SMAPE 점수: 3.3198

--- 🎯 예측 오차가 가장 큰 건물 Top 5 ---
    building_id      smape
45            7  17.354571
61           23  11.226790
38          100   7.780162
92           54   7.548164
87           49   7.513096

--- ✅ 예측 오차가 가장 작은 건물 Top 5 ---
    building_id     smape
89           51  0.525966
95           57  0.406823
94           56  0.389547
74           36  0.292209
73           35  0.199788


In [ ]:
test_whole = pd.read_parquet(data_path/'test_whole_data.parquet')
test_whole.head()

,building_id,building_type,datetime_,temperature_celsius,rainfall_mm,windspeed_mps,humidity_percent,total_area_m2,cooling_area_m2,solar_panel_capacity_kw,ess_storage_capacity_kwh,pcs_capacity_kw
0,1,호텔,2024-08-25 00:00:00,26.5,0.0,0.7,80.0,82912.71,77586.0,NaN,NaN,NaN
1,1,호텔,2024-08-25 01:00:00,26.1,0.0,0.0,80.0,82912.71,77586.0,NaN,NaN,NaN
2,1,호텔,2024-08-25 02:00:00,25.9,0.0,0.3,83.0,82912.71,77586.0,NaN,NaN,NaN
3,1,호텔,2024-08-25 03:00:00,25.7,0.0,1.1,83.0,82912.71,77586.0,NaN,NaN,NaN
4,1,호텔,2024-08-25 04:00:00,25.5,0.0,1.0,86.0,82912.71,77586.0,NaN,NaN,NaN


In [94]:
# Cell 1: 테스트 데이터 전처리

# 1. 결측치(NaN) 처리
cols_to_fill_zero = ['solar_panel_capacity_kw', 'ess_storage_capacity_kwh', 'pcs_capacity_kw']
test_whole[cols_to_fill_zero] = test_whole[cols_to_fill_zero].fillna(0)

# 2. 시간 기반 피처 생성
test_whole['datetime_'] = pd.to_datetime(test_whole['datetime_'])
test_whole['month'] = test_whole['datetime_'].dt.month
test_whole['hour'] = test_whole['datetime_'].dt.hour
test_whole['dayofweek'] = test_whole['datetime_'].dt.dayofweek
test_whole['is_weekend'] = test_whole['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)

# 3. 주기성 피처(사인/코사인) 생성
cyclical_features = ['hour', 'dayofweek', 'month']
for feature in cyclical_features:
    max_val = test_whole[feature].max()
    test_whole[feature + '_sin'] = np.sin(2 * np.pi * test_whole[feature] / (max_val + 1))
    test_whole[feature + '_cos'] = np.cos(2 * np.pi * test_whole[feature] / (max_val + 1))

# 4. Lag, Rolling, 파생 피처 등 (훈련 데이터에 적용했던 모든 추가 피처 생성)
# (이전 단계에서 train_whole_data에 적용했던 코드와 동일하게 적용해야 합니다)
# 예시: test_whole = test_whole.sort_values(by=['building_id', 'datetime_']).reset_index(drop=True)
#      test_whole['temp_rolling_mean_7d'] = ... 

# 5. 범주형 피처 원-핫 인코딩
test_whole = pd.get_dummies(test_whole, columns=['building_type'], prefix='building')

print("✅ 테스트 데이터 전처리 완료!")

✅ 테스트 데이터 전처리 완료!


In [95]:
# Cell 2: 훈련/테스트 데이터 컬럼 맞추기

# 'features' 리스트는 이전에 훈련 데이터를 준비할 때 사용했던 피처 목록입니다.
# features = X_train.columns.tolist() 

# 훈련 데이터에만 있고 테스트 데이터에는 없는 컬럼을 찾아서 0으로 채워줍니다.
missing_cols = set(features) - set(test_whole.columns)
for c in missing_cols:
    test_whole[c] = 0

# 훈련 데이터의 피처 순서와 동일하게 테스트 데이터의 컬럼 순서를 맞춥니다.
X_test_final = test_whole[features]

print("✅ 훈련 데이터와 테스트 데이터의 컬럼을 일치시켰습니다.")
print(f"최종 테스트 데이터의 형태: {X_test_final.shape}")

✅ 훈련 데이터와 테스트 데이터의 컬럼을 일치시켰습니다.
최종 테스트 데이터의 형태: (16800, 32)


In [96]:
# Cell 1: 최종 모델 학습 및 '피처 목록' 저장

import lightgbm as lgb
from tqdm import tqdm

# 학습된 모델과 '피처 목록'을 함께 저장할 빈 딕셔너리 생성
models = {}

for building_id in tqdm(range(1, 101), desc="최종 건물별 모델 학습"):
    train_building = train_preprocessed_df[train_preprocessed_df['building_id'] == building_id].copy()
    if len(train_building) == 0:
        continue
        
    target = 'power_consumption_kwh'
    # ✅ 이 'features' 리스트가 각 모델의 고유한 설계도입니다.
    features = [col for col in train_building.columns if col not in [target, 'datetime_', 'building_id', 'building_type_original']]
    
    X_train_final = train_building[features]
    y_train_final = train_building[target]
    
    model = lgb.LGBMRegressor(random_state=42)
    model.fit(X_train_final, y_train_final)
    
    # ✅ 모델과 함께, 학습에 사용된 'features' 리스트를 함께 저장합니다.
    models[building_id] = {'model': model, 'features': features}

print(f"\n✅ 모든 건물에 대한 최종 모델 학습 및 저장이 완료되었습니다! 총 {len(models)}개의 모델이 저장되었습니다.")

최종 건물별 모델 학습:   0%|          | 0/100 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000160 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1509
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 16
[LightGBM] [Info] Start training from score 5340.317489
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000117 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1509
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 16
[LightGBM] [Info] Start training from score 1676.242796
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000140 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1509
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 16
[LightGBM] [Info] Start 

최종 건물별 모델 학습:   3%|▎         | 3/100 [00:00<00:03, 27.86it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000120 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1509
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 16
[LightGBM] [Info] Start training from score 885.166882
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000116 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1509
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 16
[LightGBM] [Info] Start training from score 4542.198822
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000134 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1509
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 16
[LightGBM] [Info] Start t

최종 건물별 모델 학습:   6%|▌         | 6/100 [00:00<00:03, 28.06it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000108 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1260
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 15
[LightGBM] [Info] Start training from score 1748.916000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000093 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1260
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 15
[LightGBM] [Info] Start training from score 2362.418352
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000115 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1509
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 16
[LightGBM] [Info] Start 

최종 건물별 모델 학습:  99%|█████████▉| 99/100 [00:03<00:00, 29.93it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000094 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1499
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 16
[LightGBM] [Info] Start training from score 1814.858294


최종 건물별 모델 학습: 100%|██████████| 100/100 [00:03<00:00, 30.10it/s]


✅ 모든 건물에 대한 최종 모델 학습 및 저장이 완료되었습니다! 총 100개의 모델이 저장되었습니다.


In [97]:
# Cell 2: 저장된 '피처 목록'을 사용하여 예측 수행

all_predictions = []

for building_id in tqdm(range(1, 101), desc="건물별 예측 진행률"):
    
    # 해당 건물의 테스트 데이터 선택
    X_test_building_raw = X_test_final[X_test_final['building_id'] == building_id]
    
    # 딕셔너리에서 해당 건물의 모델 정보(모델 + 피처 리스트)를 불러옵니다.
    model_info = models.get(building_id)
    
    if model_info is not None and not X_test_building_raw.empty:
        model = model_info['model']
        train_features = model_info['features'] # ✅ 학습 시 사용했던 피처 목록을 불러옵니다.
        
        # ✅ 테스트 데이터의 컬럼을 학습 시 사용했던 피처 목록에 정확히 맞춥니다.
        #    - 없는 컬럼은 0으로 채워지고, 순서도 동일하게 정렬됩니다.
        X_test_building_aligned = X_test_building_raw.reindex(columns=train_features, fill_value=0)
        
        # 정렬된 데이터로 예측 수행
        preds = model.predict(X_test_building_aligned)
        
        prediction_df = pd.DataFrame(preds, index=X_test_building_aligned.index, columns=['answer'])
        all_predictions.append(prediction_df)

# 모든 예측 결과를 하나로 합칩니다.
final_predictions_df = pd.concat(all_predictions).sort_index()

print("\n✅ 모든 건물에 대한 예측이 완료되었습니다!")

건물별 예측 진행률: 100%|██████████| 100/100 [00:00<00:00, 787.58it/s]


✅ 모든 건물에 대한 예측이 완료되었습니다!


In [ ]:
# Cell 2: 최종 예측 및 제출 파일 생성

# num_date_time 정보가 테스트 데이터에 있는지 확인하고, 없다면 생성합니다.
# 이 컬럼은 최종 제출 파일의 ID 역할을 합니다.
if 'num_date_time' not in test_whole.columns:
    test_whole['num_date_time'] = test_whole['building_id'].astype(str) + '_' + test_whole['datetime_'].dt.strftime('%Y%m%d %H')

all_predictions = []

for building_id in tqdm(range(1, 101), desc="건물별 최종 예측 진행률"):
    
    # 딕셔너리에서 해당 건물의 모델 정보(모델 + 피처 리스트)를 불러옵니다.
    model_info = models.get(building_id)
    
    if model_info:
        model = model_info['model']
        train_features = model_info['features'] # 학습 시 사용했던 피처 목록
        
        # 해당 건물의 테스트 데이터 선택
        X_test_building_raw = test_whole[test_whole['building_id'] == building_id]
        
        if not X_test_building_raw.empty:
            # 테스트 데이터의 컬럼을 학습 시 사용했던 피처 목록에 정확히 맞춥니다.
            # 훈련 시에는 있었지만 테스트에는 없는 컬럼은 0으로 채워집니다.
            X_test_building_aligned = X_test_building_raw.reindex(columns=train_features, fill_value=0)
            
            # 정렬된 데이터로 예측 수행
            preds = model.predict(X_test_building_aligned)
            
            # 예측 결과를 num_date_time과 함께 저장
            prediction_df = pd.DataFrame({
                'num_date_time': X_test_building_raw['num_date_time'],
                'answer': preds
            })
            all_predictions.append(prediction_df)

# 모든 예측 결과를 하나로 합칩니다.
final_predictions_df = pd.concat(all_predictions)

# sample_submission.csv 파일을 불러와서 예측 결과로 채웁니다.
submission_df = pd.read_csv(data_path/'sample_submission.csv') 
submission_df = submission_df.drop(columns='answer') # 기존 answer 컬럼 삭제

# num_date_time을 기준으로 예측 결과를 병합합니다.
submission_df = pd.merge(submission_df, final_predictions_df, on='num_date_time', how='left')

# 최종 제출 파일을 저장합니다.
submission_df.to_csv('final_submission.csv', index=False)

print(f"\n🎉 최종 제출 파일 'final_submission.csv' 생성이 완료되었습니다!")
print(submission_df.head())

건물별 최종 예측 진행률: 100%|██████████| 100/100 [00:00<00:00, 695.89it/s]



🎉 최종 제출 파일 'final_submission.csv' 생성이 완료되었습니다!
   num_date_time       answer
0  1_20240825 00  5130.106048
1  1_20240825 01  5117.551566
2  1_20240825 02  4925.970095
3  1_20240825 03  4611.312433
4  1_20240825 04  4406.319391
